# `JSONDataSource` - Lecture et écriture de fichiers JSON

**Module :** `kadi.kidas.sources.json_source`  
**Classe :** `JSONDataSource`

---

## Introduction

`JSONDataSource` gère les fichiers JSON et dictionnaires Python tels qu'ils sont retournés par les APIs agricoles (FAO, WFP VAM, MAEP). Sa fonctionnalité principale est **l'aplatissement automatique** des structures imbriquées :

```
{"location": {"lat": 9.3, "lon": 2.4}, "crop": "maize"}
→ {"location.lat": 9.3, "location.lon": 2.4, "crop": "maize"}
```

La source peut être un **fichier sur le disque** ou un **dictionnaire Python en mémoire**.

## 1. Création des données de démonstration

In [1]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath('../../'))

import os
import json
import tempfile
import pandas as pd

from kadi.kidas.sources.json_source import JSONDataSource

# --- Fichier JSON simple (liste d'enregistrements) ---
donnees_json = [
    {"culture": "maïs", "marche": "Dantokpa", "prix_xof": 350, "date": "2024-01-15"},
    {"culture": "niébé", "marche": "Parakou", "prix_xof": 500, "date": "2024-01-16"},
    {"culture": "igname", "marche": "Bohicon", "prix_xof": 250, "date": "2024-01-17"},
]

fichier_json = os.path.join(tempfile.gettempdir(), 'prix_marches.json')
with open(fichier_json, 'w', encoding='utf-8') as f:
    json.dump(donnees_json, f, ensure_ascii=False, indent=2)

# --- JSON imbriqué (réponse API réelle) ---
reponse_api = {
    "metadata": {"source": "WFP VAM", "date": "2024-01-01", "version": "2.0"},
    "data": [
        {
            "location": {"country": "Benin", "city": "Cotonou", "lat": 6.366, "lon": 2.437},
            "crop": "maize",
            "prices": {"retail": 350, "wholesale": 280, "currency": "XOF"},
        },
        {
            "location": {"country": "Benin", "city": "Parakou", "lat": 9.337, "lon": 2.629},
            "crop": "cowpea",
            "prices": {"retail": 500, "wholesale": 430, "currency": "XOF"},
        },
    ],
}

fichier_imbrique = os.path.join(tempfile.gettempdir(), 'api_wfp.json')
with open(fichier_imbrique, 'w', encoding='utf-8') as f:
    json.dump(reponse_api, f, ensure_ascii=False, indent=2)

print(f"Fichier JSON simple    : {fichier_json}")
print(f"Fichier JSON imbriqué  : {fichier_imbrique}")

Fichier JSON simple    : /tmp/prix_marches.json
Fichier JSON imbriqué  : /tmp/api_wfp.json


#### Ce que produit cette cellule

```
Fichier JSON simple    : /tmp/prix_marches.json
Fichier JSON imbriqué  : /tmp/api_wfp.json
```

Deux fichiers JSON sont créés dans le dossier temporaire du système. Ils représentent deux structures très différentes, toutes deux courantes dans les APIs agricoles.

**Fichier simple** (`prix_marches.json`) — une liste de 3 enregistrements plats, chacun avec 4 champs directs :

```json
[
  {"culture": "maïs",   "marche": "Dantokpa", "prix_xof": 350, "date": "2024-01-15"},
  {"culture": "niébé",  "marche": "Parakou",  "prix_xof": 500, "date": "2024-01-16"},
  {"culture": "igname", "marche": "Bohicon",  "prix_xof": 250, "date": "2024-01-17"}
]
```

**Fichier imbriqué** (`api_wfp.json`) — une réponse typique de l'API WFP VAM avec plusieurs niveaux d'imbrication : métadonnées, données géographiques (`location`), type de culture (`crop`) et prix détaillés (`prices`). Ce format est volontairement complexe pour illustrer l'aplatissement automatique de `JSONDataSource`.

```json
{
    "metadata": {"source": "WFP VAM", "date": "2024-01-01", "version": "2.0"},
    "data": [
        {
            "location": {"country": "Benin", "city": "Cotonou", "lat": 6.366, "lon": 2.437},
            "crop": "maize",
            "prices": {"retail": 350, "wholesale": 280, "currency": "XOF"},
        },
        {
            "location": {"country": "Benin", "city": "Parakou", "lat": 9.337, "lon": 2.629},
            "crop": "cowpea",
            "prices": {"retail": 500, "wholesale": 430, "currency": "XOF"},
        },
    ],
}
```

L'option `ensure_ascii=False` dans `json.dump()` garantit que les caractères accentués (`ï`, `é`) sont écrits tels quels dans le fichier, sans être convertis en séquences d'échappement.

## 2. Initialisation de `JSONDataSource`

```python
JSONDataSource(file_path_or_dict: str | dict)
```

Accepte soit un **chemin de fichier** (str), soit un **dictionnaire Python** directement.

In [2]:
# Depuis un fichier
source_fichier = JSONDataSource(fichier_json)
print(repr(source_fichier))

JSONDataSource(source_path='/tmp/prix_marches.json', source_type='json', encoding='utf-8')


#### Ce que produit cette cellule

```
JSONDataSource(source_path='/tmp/prix_marches.json', source_type='json', encoding='utf-8')
```

La représentation de l'objet montre trois champs :

| Champ | Valeur | Ce que ça signifie |
|---|---|---|
| `source_path` | `/tmp/prix_marches.json` | Le chemin complet vers le fichier JSON sur le disque. |
| `source_type` | `json` | Le type de source — toujours `json` pour cette classe. |
| `encoding` | `utf-8` | JSON est par définition du texte Unicode (RFC 8259). L'encodage UTF-8 est connu sans analyse, contrairement aux CSV. |

La source est initialisée avec un chemin de fichier : `JSONDataSource` reconnaît qu'il s'agit d'un fichier et mémorise son chemin pour les opérations de lecture et écriture ultérieures.

In [3]:
# Depuis un dictionnaire Python (source en mémoire)
dict_source = {"culture": "maïs", "prix_xof": 350, "commune": "Cotonou"}
source_dict = JSONDataSource(dict_source)
print(repr(source_dict))

JSONDataSource(source_path='<dict_en_memoire>', source_type='json', encoding='utf-8')


#### Ce que produit cette cellule

```
JSONDataSource(source_path='<dict_en_memoire>', source_type='json', encoding='utf-8')
```

Lorsqu'on passe un **dictionnaire Python** directement (plutôt qu'un chemin de fichier), la classe affiche `<dict_en_memoire>` comme valeur de `source_path`.

C'est un marqueur interne qui signale que les données vivent **en mémoire** et n'ont pas de chemin physique sur le disque.

**Différence clé entre les deux modes :**

| Mode | `source_path` affiché | Origine des données |
|---|---|---|
| Depuis un fichier | `/tmp/prix_marches.json` | Fichier lu depuis le disque. |
| Depuis un dict | `<dict_en_memoire>` | Données déjà disponibles en mémoire (ex : réponse d'API). |

Le mode dictionnaire est particulièrement pratique lorsqu'on reçoit une réponse d'API (`requests.get(...).json()`) et qu'on veut la traiter directement sans passer par un fichier intermédiaire.

## 3. `validate_connection()` - Vérifier l'accessibilité

In [4]:
# Fichier existant
print(f"Fichier accessible    : {JSONDataSource(fichier_json).validate_connection()}")

# Fichier absent
print(f"Fichier absent        : {JSONDataSource('/absent.json').validate_connection()}")

# Dictionnaire en mémoire (toujours accessible)
print(f"Dict mémoire          : {JSONDataSource({'a': 1}).validate_connection()}")

Le fichier JSON '/absent.json' n'existe pas ou n'est pas lisible.


Fichier accessible    : True
Fichier absent        : False
Dict mémoire          : True


#### Ce que produit cette cellule

```
[erreur standard] Le fichier JSON '/absent.json' n'existe pas ou n'est pas lisible.
[sortie standard] Fichier accessible    : True
[sortie standard] Fichier absent        : False
[sortie standard] Dict mémoire          : True
```

Trois appels à `validate_connection()` illustrent les trois situations possibles :

| Source testée | Résultat | Explication |
|---|---|---|
| `/tmp/prix_marches.json` | `True` | Le fichier existe et est lisible. |
| `/absent.json` | `False` | Le fichier n'existe pas — un avertissement est écrit sur l'erreur standard, puis `False` est retourné sans planter. |
| `{'a': 1}` (dict) | `True` | Un dictionnaire en mémoire est toujours considéré comme accessible — pas de fichier à vérifier. |

**Comportement du message d'erreur :**

Le message `... n'existe pas ou n'est pas lisible.` apparaît en rouge dans Jupyter car il est envoyé sur l'erreur standard, non sur la sortie standard. Il est affiché avant les lignes `print()` car Python traite ces deux canaux séparément.

Cette logique différenciée (fichier vs dictionnaire) est propre à `JSONDataSource` — les autres sources (`CSVDataSource`, `ExcelDataSource`) n'acceptent que des chemins de fichiers.

## 4. `flatten_json()` - Aplatissement d'un JSON imbriqué

Transforme les clés imbriquées en clés à **notation pointée**.

In [5]:
source = JSONDataSource(fichier_json)

# Exemple simple
entree_simple = {"location": {"lat": 9.3, "lon": 2.4}, "crop": "maize"}
aplati = source.flatten_json(entree_simple)
print("Entrée :")
print(f"  {entree_simple}")
print("\nAplati :")
for cle, val in aplati.items():
    print(f"  {cle} = {val}")

Entrée :
  {'location': {'lat': 9.3, 'lon': 2.4}, 'crop': 'maize'}

Aplati :
  location.lat = 9.3
  location.lon = 2.4
  crop = maize


#### Ce que produit cette cellule

```
Entrée :
  {'location': {'lat': 9.3, 'lon': 2.4}, 'crop': 'maize'}

Aplati :
  location.lat = 9.3
  location.lon = 2.4
  crop = maize
```

`flatten_json()` transforme un dictionnaire imbriqué en dictionnaire plat en joignant les clés parentes et enfants avec un point.

**Transformation clé par clé :**

| Avant (structure imbriquée) | Après (structure plate) |
|---|---|
| `location` → `{lat: 9.3, lon: 2.4}` | Supprimé (remplacé par ses enfants) |
| `location.lat` | `location.lat = 9.3` |
| `location.lon` | `location.lon = 2.4` |
| `crop` | `crop = maize` (inchangé, déjà plat) |

Les clés déjà au premier niveau (`crop`) ne sont pas modifiées. Seules les clés qui contiennent un sous-dictionnaire sont décomposées. Le résultat est un dictionnaire **directement convertissable en ligne de tableau**, ce qui est l'objectif : produire un DataFrame propre depuis un JSON imbriqué.

In [6]:
# Exemple avec liste imbriquée
entree_liste = {
    "commune": "Cotonou",
    "cultures": ["maïs", "niébé", "igname"],
    "stats": {"total": 3, "actif": True},
}
aplati_complet = source.flatten_json(entree_liste)
for cle, val in aplati_complet.items():
    print(f"  {cle} = {val}")

  commune = Cotonou
  cultures.0 = maïs
  cultures.1 = niébé
  cultures.2 = igname
  stats.total = 3
  stats.actif = True


#### Ce que produit cette cellule

```
  commune = Cotonou
  cultures.0 = maïs
  cultures.1 = niébé
  cultures.2 = igname
  stats.total = 3
  stats.actif = True
```

Cette cellule montre que `flatten_json()` gère également les **listes imbriquées**, pas seulement les dictionnaires.

**Transformation détaillée :**

| Structure d'entrée | Résultat aplati | Règle appliquée |
|---|---|---|
| `"commune": "Cotonou"` | `commune = Cotonou` | Valeur simple, aucun changement. |
| `"cultures": ["maïs", "niébé", "igname"]` | `cultures.0`, `cultures.1`, `cultures.2` | Chaque élément de la liste reçoit son **index numérique** comme suffixe. |
| `"stats": {"total": 3, "actif": True}` | `stats.total = 3`, `stats.actif = True` | Dictionnaire imbriqué → notation pointée standard. |

La numérotation des éléments de liste (`cultures.0`, `cultures.1`, etc.) est une convention qui permet de les distinguer dans le DataFrame final. C'est une approche pratique, bien que dans la plupart des cas on préférera normaliser ces listes en lignes séparées lors d'un traitement ultérieur.

In [7]:
# Séparateur personnalisé ('_' au lieu de '.')
aplati_underscore = source.flatten_json(entree_simple, separateur='_')
print("Avec séparateur '_' :")
for cle, val in aplati_underscore.items():
    print(f"  {cle} = {val}")

Avec séparateur '_' :
  location_lat = 9.3
  location_lon = 2.4
  crop = maize


#### Ce que produit cette cellule

```
Avec séparateur '_' :
  location_lat = 9.3
  location_lon = 2.4
  crop = maize
```

Le paramètre `separateur` permet de choisir le caractère qui relie les niveaux dans les noms de colonnes.

**Comparaison des deux séparateurs :**

| Séparateur | Résultat pour `location.lat` | Cas d'usage |
|---|---|---|
| `.` (par défaut) | `location.lat` | Lecture humaine, style Python naturel. |
| `_` | `location_lat` | Noms de colonnes SQL-compatibles, sans caractère spécial. |

Le séparateur `_` est particulièrement utile lorsque les noms de colonnes résultants seront utilisés directement dans des requêtes SQL ou des noms de colonnes pandas sans guillemets, car le point est un opérateur d'attribut en Python et peut causer des ambiguïtés dans certains contextes.

## 5. `read()` - Lecture et conversion en DataFrame

In [8]:
# Lecture simple (JSON plat)
source = JSONDataSource(fichier_json)
df = source.read()
print(f"JSON simple : {len(df)} lignes × {len(df.columns)} colonnes")
df

JSON simple : 3 lignes × 4 colonnes


,culture,marche,prix_xof,date
0,maïs,Dantokpa,350,2024-01-15
1,niébé,Parakou,500,2024-01-16
2,igname,Bohicon,250,2024-01-17


#### Ce que produit cette cellule

```
JSON simple : 3 lignes × 4 colonnes
```

Suivi du tableau avec 3 lignes et 4 colonnes.

**Analyse du tableau :**

| Colonne | Type inféré | Observation |
|---|---|---|
| `culture` | texte | Noms de cultures avec accents, lus correctement grâce à l'UTF-8. |
| `marche` | texte | Marchés béninois : Dantokpa (Cotonou), Parakou, Bohicon. |
| `prix_xof` | entier | Prix en Franc CFA, inférés comme `int64` car sans décimales. |
| `date` | texte | Dates au format ISO (`YYYY-MM-DD`) lues comme chaînes — à convertir avec `pd.to_datetime()` si besoin. |

**Ce que fait `read()` pour un JSON plat :**
1. Charge le fichier avec `json.load()`.
2. Détecte que la structure est une liste de dictionnaires (format tabulaire).
3. Appelle directement `pd.DataFrame()` sur la liste.
4. Met à jour `source.last_read` avec l'heure courante.

Quand le JSON est déjà plat (aucune imbrication), l'aplatissement est sans effet et le résultat est identique à un `pd.read_json()` standard.

In [9]:
# Lecture depuis un dict Python
source_dict = JSONDataSource({"culture": "maïs", "prix": 350, "marche": "Dantokpa"})
df_dict = source_dict.read()
df_dict

,culture,prix,marche
0,maïs,350,Dantokpa


#### Ce que produit cette cellule

Un tableau avec **1 seule ligne** et **3 colonnes** (`culture`, `prix`, `marche`).

**Analyse :**

Lorsque la source est un **dictionnaire simple** (pas une liste), `read()` le convertit en un DataFrame d'une seule ligne. Chaque clé du dictionnaire devient le nom d'une colonne, et l'unique valeur associée devient la cellule de la ligne 0.

| Clé du dict | Valeur | Colonne pandas | Valeur ligne 0 |
|---|---|---|---|
| `culture` | `"maïs"` | `culture` | `maïs` |
| `prix` | `350` | `prix` | `350` |
| `marche` | `"Dantokpa"` | `marche` | `Dantokpa` |

Ce cas d'usage correspond au traitement d'un enregistrement unique — par exemple, la réponse d'une API qui retourne les données d'un seul marché ou d'une seule culture.

In [10]:
# Lecture d'un JSON imbriqué AVEC aplatissement (flatten=True, par défaut)
source_imb = JSONDataSource(reponse_api)
df_aplati = source_imb.read(flatten=True)
print(f"JSON imbriqué aplati : {len(df_aplati)} lignes × {len(df_aplati.columns)} colonnes")
df_aplati

JSON imbriqué aplati : 1 lignes × 19 colonnes


,metadata.source,metadata.date,metadata.version,data.0.location.country,data.0.location.city,data.0.location.lat,data.0.location.lon,data.0.crop,data.0.prices.retail,data.0.prices.wholesale,data.0.prices.currency,data.1.location.country,data.1.location.city,data.1.location.lat,data.1.location.lon,data.1.crop,data.1.prices.retail,data.1.prices.wholesale,data.1.prices.currency
0,WFP VAM,2024-01-01,2.0,Benin,Cotonou,6.366,2.437,maize,350,280,XOF,Benin,Parakou,9.337,2.629,cowpea,500,430,XOF


#### Ce que produit cette cellule

```
JSON imbriqué aplati : 1 lignes × 19 colonnes
```

Suivi d'un tableau très large avec **1 seule ligne** et **19 colonnes** aux noms à notation pointée.

**Pourquoi 1 ligne et 19 colonnes ?**

La réponse API `reponse_api` est un dictionnaire avec 2 clés de premier niveau (`metadata` et `data`). L'aplatissement transforme l'**ensemble de la structure** en une seule ligne, chaque valeur terminale (feuille) devenant une colonne.

**Décompte des 19 colonnes :**

| Préfixe | Colonnes produites | Nombre |
|---|---|---|
| `metadata.*` | `source`, `date`, `version` | 3 |
| `data.0.location.*` | `country`, `city`, `lat`, `lon` | 4 |
| `data.0.*` | `crop` + `prices.retail`, `prices.wholesale`, `prices.currency` | 4 |
| `data.1.location.*` | `country`, `city`, `lat`, `lon` | 4 |
| `data.1.*` | `crop` + `prices.retail`, `prices.wholesale`, `prices.currency` | 4 |
| **Total** | | **19** |

**Limites de cette représentation :** Avec `flatten=True` sur une réponse API volumineuse, le DataFrame devient très large et peu lisible. C'est pourquoi, en pratique, on préfère extraire uniquement la clé `data` et traiter chaque entrée séparément avant de concaténer les résultats.

In [11]:
# Lecture SANS aplatissement (flatten=False)
df_non_aplati = source_imb.read(flatten=False)
print(f"JSON imbriqué non aplati : {len(df_non_aplati)} lignes × {len(df_non_aplati.columns)} colonnes")
df_non_aplati.head()

JSON imbriqué non aplati : 1 lignes × 4 colonnes


,data,metadata.source,metadata.date,metadata.version
0,"[{'location': {'country': 'Benin', 'city': 'Co...",WFP VAM,2024-01-01,2.0


#### Ce que produit cette cellule

```
JSON imbriqué non aplati : 1 lignes × 4 colonnes
```

Suivi d'un tableau avec **1 ligne** et **seulement 4 colonnes**.

**Comparaison `flatten=True` vs `flatten=False` sur la même source :**

| Paramètre | Colonnes | Contenu de `data` |
|---|---|---|
| `flatten=True` | 19 colonnes plates | Toutes les valeurs terminales extraites et nommées. |
| `flatten=False` | 4 colonnes | `data` contient **la liste entière** des enregistrements, non décomposée. |

**Ce qu'on voit dans le tableau :**

- **Colonne `data`** : la cellule contient `[{'location': {'country': 'Benin', 'city': 'Co...` — c'est la liste complète des 2 enregistrements, affichée tronquée par pandas. La structure imbriquée est conservée intacte.
- **Colonnes `metadata.*`** : même avec `flatten=False`, la clé `metadata` est partiellement aplatie (ses sous-clés sont visibles). Cela vient du fait que `metadata` est un simple dictionnaire et non une liste — pandas le développe automatiquement lors de la construction du DataFrame.

`flatten=False` est utile quand on veut d'abord inspecter la structure avant de décider quelle partie traiter, ou quand la liste `data` doit être traitée séparément avec `pd.json_normalize()`.

## 6. `write()` - Écriture vers un fichier JSON

In [12]:
df_a_ecrire = pd.DataFrame({
    'culture': ['fonio', 'soja'],
    'prix_xof': [420, 600],
    'marche': ['Malanville', 'Natitingou'],
})

fichier_sortie = os.path.join(tempfile.gettempdir(), 'sortie.json')
source_sortie = JSONDataSource(fichier_sortie)

# Format 'records' (liste de dictionnaires)
succes = source_sortie.write(df_a_ecrire, orient='records')
print(f"Écriture réussie : {succes}")

# Lecture du fichier écrit pour vérification
with open(fichier_sortie, encoding='utf-8') as f:
    print(f.read())

Écriture réussie : True
[
  {
    "culture":"fonio",
    "prix_xof":420,
    "marche":"Malanville"
  },
  {
    "culture":"soja",
    "prix_xof":600,
    "marche":"Natitingou"
  }
]


#### Ce que produit cette cellule

```
Écriture réussie : True
[
  {
    "culture":"fonio",
    "prix_xof":420,
    "marche":"Malanville"
  },
  {
    "culture":"soja",
    "prix_xof":600,
    "marche":"Natitingou"
  }
]
```

**Déroulement en deux temps :**

**Écriture** — `write(df, orient='records')` convertit le DataFrame en une **liste de dictionnaires** (un dictionnaire par ligne), puis l'écrit dans `sortie.json`. La valeur `True` confirme que l'opération a réussi.

**Vérification** — Le fichier est relu directement avec `open()` et affiché tel quel. Le résultat est un JSON bien formé, lisible et indenté.

**Pourquoi `orient='records'` ?**

Le paramètre `orient` contrôle la structure JSON produite. `'records'` est le format le plus naturel pour des données tabulaires, car chaque objet JSON correspond à une ligne du tableau avec ses propres clés.

| Culture | Prix | Marché | Contexte géographique |
|---|---|---|---|
| fonio | 420 XOF | Malanville | Ville frontalière au nord du Bénin (frontière avec le Niger). |
| soja | 600 XOF | Natitingou | Ville du département de l'Atacora, au nord-ouest du Bénin. |

Ce format JSON en sortie est directement lisible par n'importe quelle API ou service web qui consomme du JSON tabulaire.

## 7. `get_metadata()` - Métadonnées de la source

In [13]:
source = JSONDataSource(fichier_json)
source.read()
meta = source.get_metadata()
for cle, valeur in meta.items():
    print(f"  {cle:15s} : {valeur}")

  source_path     : /tmp/prix_marches.json
  source_type     : json
  is_file         : True
  size_kb         : 0.31
  last_read       : 2026-07-07T18:20:36.615468


#### Ce que produit cette cellule

```
  source_path     : /tmp/prix_marches.json
  source_type     : json
  is_file         : True
  size_kb         : 0.31
  last_read       : 2026-07-07T18:20:36.615468
```

`get_metadata()` retourne 5 champs décrivant l'état de la source après lecture.

**Détail de chaque champ :**

| Champ | Valeur | Ce que ça signifie |
|---|---|---|
| `source_path` | `/tmp/prix_marches.json` | Chemin du fichier JSON sur le disque. |
| `source_type` | `json` | Type de source — toujours `json` pour cette classe. |
| `is_file` | `True` | Confirme que la source est un **fichier** (et non un dictionnaire en mémoire). Pour un dict, cette valeur serait `False`. |
| `size_kb` | `0.31` | Taille du fichier en kilo-octets (310 octets environ — un petit fichier de 3 enregistrements). |
| `last_read` | `2026-07-07T18:20:36...` | Horodatage ISO 8601 de la dernière lecture, mis à jour par `source.read()` juste avant cet appel. |

**Différence avec `ExcelDataSource` et `CSVDataSource` :**

`JSONDataSource` expose le champ `is_file` (absent des deux autres sources), car c'est la seule source capable d'accepter à la fois un fichier et un dictionnaire en mémoire. Ce champ permet de savoir exactement ce qu'on manipule sans inspecter le code.

## Récapitulatif des méthodes

| Méthode | Description |
|---|---|
| `validate_connection()` | Vérifie le fichier (ou retourne True pour dict mémoire) |
| `flatten_json(obj, sep)` | Aplatit récursivement un JSON imbriqué (notation pointée) |
| `read(flatten)` | Lit et convertit le JSON en DataFrame |
| `write(data, orient)` | Écrit un DataFrame au format JSON |
| `get_metadata()` | Retourne chemin, type, taille, last_read |